# Introduction to Homecastr Forecasts

**Goal:** Pull your first probabilistic home value forecast and understand what P10, P50, and P90 mean.

Homecastr produces Bayesian ensemble forecasts — not a single price estimate, but a *distribution* over outcomes. This notebook walks through:

1. Connecting to the API
2. Fetching a forecast by address
3. Visualizing the probability bands
4. Understanding the reliability score

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/homecastr/homecastr-cookbook/blob/main/examples/getting_started/01_introduction.ipynb)

In [ ]:
# Install the SDK
# !pip install -U homecastr matplotlib

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from homecastr import HomecastrClient

# The demo key works out of the box for exploration.
# Replace with your key from homecastr.com for higher rate limits.
api_key = os.getenv("HOMECASTR_API_KEY", "hc_demo_public_readonly")
client = HomecastrClient(api_key)

print(client.ping())

## 1. Fetch a Forecast by Address

Pass any US street address. We recommend including city and state for best geocoding accuracy.

In [ ]:
ADDRESS = "4521 Westheimer Rd Houston TX 77027"

result = client.forecast.by_address.retrieve(ADDRESS, year=2028)

print(f"Address:          {result['address']}")
print(f"Current value:    ${result['current_value']:,}")
print()
print(f"2028 forecast (P10): ${result['forecasts']['p10']:,}  — downside scenario")
print(f"2028 forecast (P50): ${result['forecasts']['p50']:,}  — median expectation")
print(f"2028 forecast (P90): ${result['forecasts']['p90']:,}  — upside scenario")
print()
print(f"Appreciation:     {result['appreciation_pct']:.1f}%")
print(f"Reliability:      {result['reliability']:.2f}  (0 = low confidence, 1 = high)")
print(f"Comparables:      {result['property_count']} properties in training data")

## 2. Visualize the Fan Chart

The `fan_chart` field contains the full probability distribution across all forecast horizons (1–5 years).

In [ ]:
import pandas as pd

fan = pd.DataFrame(result["fan_chart"])

fig, ax = plt.subplots(figsize=(10, 5))

# Probability bands
ax.fill_between(fan["year"], fan["p10"] / 1e3, fan["p90"] / 1e3,
                alpha=0.15, color="steelblue", label="P10–P90 range")
ax.fill_between(fan["year"], fan["p25"] / 1e3, fan["p75"] / 1e3,
                alpha=0.3, color="steelblue", label="P25–P75 range")
ax.plot(fan["year"], fan["p50"] / 1e3, color="steelblue", linewidth=2, label="P50 (median)")

# Current value
ax.axhline(result["current_value"] / 1e3, color="gray", linestyle="--", linewidth=1, label="Current value")

ax.set_title(f"Homecastr Forecast Fan Chart\n{result['address']}", fontsize=13)
ax.set_xlabel("Year")
ax.set_ylabel("Value ($000s)")
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:.0f}k"))
plt.tight_layout()
plt.show()

print("\nFan chart data:")
print(fan[["year", "p10", "p25", "p50", "p75", "p90"]].to_string(index=False))

## 3. Understanding the Reliability Score

Reliability is a composite confidence score (0–1) based on:
- Number of comparable properties in the training data
- Median absolute prediction error on held-out samples
- Coefficient of variation across ensemble models
- Temporal coverage (how many years of history)

| Score | Interpretation |
|---|---|
| 0.85+ | High confidence — dense training data, low error |
| 0.70–0.85 | Moderate confidence — typical for suburban areas |
| 0.50–0.70 | Lower confidence — use P10/P90 range for scenario planning |
| < 0.50 | Sparse data — treat forecast as directional only |

In [ ]:
reliability = result["reliability"]
p10 = result["forecasts"]["p10"]
p50 = result["forecasts"]["p50"]
p90 = result["forecasts"]["p90"]

spread_pct = (p90 - p10) / p50 * 100

print(f"Reliability score:  {reliability:.2f}")
print(f"P10/P90 spread:     {spread_pct:.1f}% of median — this is your uncertainty range")
print(f"Upside potential:   +${p90 - p50:,} ({(p90/p50 - 1)*100:.1f}% above median)")
print(f"Downside risk:      -${p50 - p10:,} ({(1 - p10/p50)*100:.1f}% below median)")

## Next Steps

- **[02_bulk_lookup.ipynb](02_bulk_lookup.ipynb)** — Retrieve forecasts for a list of addresses and export to CSV
- **[03_parcel_fan_chart.ipynb](03_parcel_fan_chart.ipynb)** — Fetch lot-level forecasts by county tax parcel ID
- **[../neighborhoods/01_h3_heatmap.ipynb](../neighborhoods/01_h3_heatmap.ipynb)** — Map opportunity scores across a city